In [30]:
import duckdb
import os

# 1. Identify the Project Root (finds 'triangle-spatial-data-engine' folder)
# This makes paths robust across different environments
root = os.path.abspath(os.path.join(os.getcwd(), "../../")) # Adjust based on notebook location
db_path = os.path.join(root, "data/triangle_engine.db")

# 2. Connect to the Single Source of Truth
con = duckdb.connect(db_path)

# 3. Standard Spatial Setup
con.execute("INSTALL spatial; LOAD spatial;")

print(f"Engine Connected: {db_path}")

Engine Connected: /workspaces/triangle-spatial-data-engine/data/triangle_engine.db


In [31]:
# Project-specific paths
proj_dir = os.path.join(root, "projects/01_food_desert_transit")
bronze_dir = os.path.join(proj_dir, "data/bronze")
silver_dir = os.path.join(proj_dir, "data/silver")

print(f"Project 01 Context Set.\nBronze: {bronze_dir}")

Project 01 Context Set.
Bronze: /workspaces/triangle-spatial-data-engine/projects/01_food_desert_transit/data/bronze


In [32]:
# Create Views using absolute paths to avoid IOErrors
con.execute(f"CREATE OR REPLACE VIEW bus_stops AS SELECT * FROM '{bronze_dir}/bus_stops.parquet'")
con.execute(f"CREATE OR REPLACE VIEW grocery_stores AS SELECT * FROM '{bronze_dir}/grocery_stores.parquet'")

print("Spatial Views Mounted.")

Spatial Views Mounted.


In [33]:
# Audit the structure of Bus Stops 
print("--- Bus Stops Schema ---")
con.execute("DESCRIBE bus_stops;").df()

--- Bus Stops Schema ---


,column_name,column_type,null,key,default,extra
0,OBJECTID,INTEGER,YES,None,None,None
1,Stop_Code,INTEGER,YES,None,None,None
2,Stop_ID,INTEGER,YES,None,None,None
3,Stop_Name,VARCHAR,YES,None,None,None
4,Street,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
120,Construction_Set,VARCHAR,YES,None,None,None
121,Bench_Install_Year,VARCHAR,YES,None,None,None
122,SWS_Pickup,VARCHAR,YES,None,None,None
123,Timepoint,VARCHAR,YES,None,None,None


In [34]:
# Audit the structure of Grocery Stores
print("\n--- Grocery Stores Schema ---")
con.execute("DESCRIBE grocery_stores;").df()


--- Grocery Stores Schema ---


,column_name,column_type,null,key,default,extra
0,type,VARCHAR,YES,None,None,None
1,id,BIGINT,YES,None,None,None
2,lat,DOUBLE,YES,None,None,None
3,lon,DOUBLE,YES,None,None,None
4,tags,"STRUCT(""addr:city"" VARCHAR, ""addr:country"" VAR...",YES,None,None,None
...,...,...,...,...,...,...
67,diet:organic,VARCHAR,YES,None,None,None
68,diet:seafood,VARCHAR,YES,None,None,None
69,diet:vegan,VARCHAR,YES,None,None,None
70,diet:vegetarian,VARCHAR,YES,None,None,None


### Data Exploration & Validation Strategy

Before performing any transformations or moving data from bronze to silver, I am conducting an audit of the raw files. A schema definition only shows the shape of the data; it does not reveal the health or the actual content.

**Approach:**

* **Individualized Inspection:** Splitting the audit into separate cells to isolate outputs and avoid information overload.
* **Bus Stop Integrity:** Verifying the legitimacy of the spatial points. I need to confirm the record counts and ensure the dataset isn't riddled with anomalies that would break future analysis.
* **Grocery Store Deconstruction:** With 72 columns, most of this dataset is noise. I need to explore the nested tags structure to identify which attributes are actually populated and relevant.

In [35]:
columns = con.table("bus_stops").columns
cols_per_row = 4

for i in range(0, len(columns), cols_per_row):
    row = columns[i:i + cols_per_row]
    print("".join(f"{name:<30}" for name in row))

OBJECTID                      Stop_Code                     Stop_ID                       Stop_Name                     
Street                        Cross_Street                  Position                      Stop_Lat                      
Stop_Lon                      Downtown                      Council_District              Sign_Crew                     
Road_Maint                    GoRaleigh                     GoTriangle                    Wolfline                      
Wake_Forest                   GoDurham                      GoCary                        Shelter                       
Shelter_Type                  Shelter_Count                 Bench                         Bench_Type                    
Bench_Count                   Trash_Can                     Trash_Can_Type                Trash_Can_Count               
Landing_Pad                   Amenity_Pad                   Pad_Size                      Bike_Rack                     
Lighting                      Li

In [36]:
columns = con.table("grocery_stores").columns
cols_per_row = 4

for i in range(0, len(columns), cols_per_row):
    row = columns[i:i + cols_per_row]
    print("".join(f"{name:<30}" for name in row))

type                          id                            lat                           lon                           
tags                          brand                         brand:wikidata                name                          
shop                          geometry                      addr:city                     addr:housenumber              
addr:postcode                 addr:state                    addr:street                   wheelchair                    
branch                        opening_hours                 phone                         ref                           
website                       addr:unit                     addr:country                  origin                        
source                        check_date                    check_date:opening_hours      butcher                       
contact:email                 contact:phone                 contact:website               cuisine                       
addr:county                   op

In [37]:
# Bus Stop Integrity
con.execute(f"""
    SELECT 
        COUNT(*) as total_records,
        COUNT(geometry) FILTER (WHERE geometry IS NOT NULL) as valid_geom,
        COUNT(*) FILTER (WHERE Stop_Name IS NULL OR Stop_Name = '') as missing_names,
        COUNT(*) FILTER (WHERE Status != 'Active') as inactive_stops,
        -- Check for "Null Island" (0,0) coordinates
        COUNT(*) FILTER (WHERE ST_X(geometry) = 0) as null_island_points
    FROM '{bronze_dir}/bus_stops.parquet';
""").df()

,total_records,valid_geom,missing_names,inactive_stops,null_island_points
0,1402,1402,0,0,0


In [38]:
# Grocery Integrity
con.execute(f"""
    SELECT 
        COUNT(*) as total_records,
        COUNT(*) FILTER (WHERE lat IS NOT NULL AND lon IS NOT NULL) as has_coords,
        COUNT(*) FILTER (WHERE lat = 0 AND lon = 0) as null_island,
        COUNT(*) FILTER (WHERE shop = 'supermarket') as supermarkets,
        COUNT(*) FILTER (WHERE shop = 'grocery') as grocery_stores,
        COUNT(*) FILTER (WHERE name IS NULL OR name = '') as unnamed
    FROM '{bronze_dir}/grocery_stores.parquet'
    WHERE shop IN ('supermarket', 'grocery');
""").df()

,total_records,has_coords,null_island,supermarkets,grocery_stores,unnamed
0,67,67,0,67,0,0


In [39]:
# 1. Check Bus Stops (Bronze)
print("--- Bus Stops Metadata ---")
display(con.execute(f"SELECT ST_AsText(geometry) as sample_geom FROM '{bronze_dir}/bus_stops.parquet' LIMIT 1").df())

# 2. Check Grocery (Bronze)
# Since this is raw OSM, we check the lat/lon values directly
print("\n--- Grocery Coordinate Ranges ---")
display(con.execute(f"""
    SELECT 
        MIN(lat) as min_lat, MAX(lat) as max_lat,
        MIN(lon) as min_lon, MAX(lon) as max_lon
    FROM '{bronze_dir}/grocery_stores.parquet'
""").df())

--- Bus Stops Metadata ---


,sample_geom
0,POINT (-78.6533558284032 35.7815446216135)



--- Grocery Coordinate Ranges ---


,min_lat,max_lat,min_lon,max_lon
0,35.702224,35.899037,-78.792064,-78.505275
